<a href="https://colab.research.google.com/github/Shahid-Ahmed-S/GenAI/blob/main/ChatBot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install transformers gradio -q
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from transformers.utils import logging
import warnings
warnings.filterwarnings('ignore')
logging.set_verbosity_error()

MODEL = 'facebook/blenderbot-400M-distill'

print('Loading chatbot model...')
tokenizer = AutoTokenizer.from_pretrained(MODEL)
model     = AutoModelForSeq2SeqLM.from_pretrained(MODEL)
print('Chatbot ready!')


Loading chatbot model...


config.json:   0%|          | 0.00/1.57k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.15k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/127k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/62.9k [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B /  730MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/355 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/347 [00:00<?, ?B/s]

Chatbot ready!


In [ ]:
def chat(user_message, history):
    '''
    user_message : what you typed
    history      : list of all messages so far
    '''
    # Add your message to history
    history.append({'role': 'user', 'content': user_message})

    # Build the full conversation text for the model
    # Each message separated by a special token
    conversation = tokenizer.sep_token.join(
        [m['content'] for m in history]
    )

    # Tokenize and generate reply
    inputs = tokenizer(
        conversation,
        return_tensors = 'pt',
        truncation     = True,
        max_length     = 128
    )
    output = model.generate(
        **inputs,
        max_new_tokens = 60
    )
    bot_reply = tokenizer.decode(output[0], skip_special_tokens=True)

    # Add bot reply to history
    history.append({'role': 'bot', 'content': bot_reply})

    return bot_reply, history

print('chat() function ready!')


chat() function ready!


In [ ]:
# Start with empty history
history = []

# Send first message
reply, history = chat('What is artificial intelligence?', history)

print('You :', 'What is artificial intelligence?')
print('Bot :', reply)


You : What is artificial intelligence?
Bot :  It is the study of human behavior and how it affects the physical, mental, social, and intellectual development of a person.


In [ ]:
# Fresh start
history = []

# Message 1 — tell it something
reply1, history = chat('My favourite sport is cricket.', history)
print('You : My favourite sport is cricket.')
print('Bot :', reply1)
print()

# Message 2 — ask a follow-up
reply2, history = chat('What sport did I just mention?', history)
print('You : What sport did I just mention?')
print('Bot :', reply2)
print()
print('The bot should remember cricket!')


You : My favourite sport is cricket.
Bot :  Cricket is a great sport. It is played on a rectangular field.

You : What sport did I just mention?
Bot :  It is a team sport that involves hitting a ball with a stick to score points.

The bot should remember cricket!


In [ ]:
# Fresh start
history = []

questions = [
    'Tell me about smart cities.',
    'What sensors are used in smart cities?',
    'How does AI help manage traffic?',
    'What are the benefits for citizens?',
]

print('Full conversation about Smart Cities:')
print('=' * 45)
print()

for i, question in enumerate(questions, 1):
    reply, history = chat(question, history)
    print(f'Turn {i}')
    print(f'You : {question}')
    print(f'Bot : {reply}')
    print()


Full conversation about Smart Cities:

Turn 1
You : Tell me about smart cities.
Bot :  Smaller cities are the best. Small towns are so much more peaceful and quiet.

Turn 2
You : What sensors are used in smart cities?
Bot :  I live in a small town and it is very quiet and peaceful. I love it.

Turn 3
You : How does AI help manage traffic?
Bot :  I'm not sure, but I do know that there are more than seventy million people in the US that live in small towns.

Turn 4
You : What are the benefits for citizens?
Bot :  It depends on the size of the city and the population density of the area.



In [ ]:
# ✏️ Change these questions!
history = []

my_questions = [
    'Tell me about machine learning.',
    'How is it different from regular programming?',
    'Can you give me a simple example?',
]

for i, question in enumerate(my_questions, 1):
    reply, history = chat(question, history)
    print(f'You : {question}')
    print(f'Bot : {reply}')
    print()


In [ ]:
import gradio as gr

# Store history between messages
memory = {'history': []}

def chat_app(user_message, chatbox):
    if not user_message.strip():
        return chatbox, ''

    # Get reply using our chat() function
    reply, memory['history'] = chat(user_message, memory['history'])

    # Add to display
    chatbox.append((user_message, reply))
    return chatbox, ''


def clear_chat():
    # Reset memory — fresh start
    memory['history'] = []
    return [], ''


with gr.Blocks(title='AI Chatbot') as app:
    gr.Markdown('# AI Chatbot — Blenderbot')
    gr.Markdown('Chat with AI! It **remembers** your whole conversation.')
    gr.Markdown('Click **Clear** to start fresh.')

    chatbox   = gr.Chatbot(label='Conversation', height=350)
    msg_input = gr.Textbox(
        label       = 'Your message',
        placeholder = 'Type anything...',
        lines       = 1
    )
    with gr.Row():
        send_btn  = gr.Button('Send',  variant='primary')
        clear_btn = gr.Button('Clear')

    send_btn.click(
        fn      = chat_app,
        inputs  = [msg_input, chatbox],
        outputs = [chatbox, msg_input]
    )
    clear_btn.click(
        fn      = clear_chat,
        outputs = [chatbox, msg_input]
    )

app.launch(share=True)


In [ ]:
app.close()
print('App closed.')
